# 08 — Vector Store
### LangChain Retrieval · Part 3 of 4

`01_Models/02_embedding_models.ipynb` found the best-matching chunk by hand: a Python list, a
for-loop, `cosine_similarity` called once per chunk. That's fine for 6 tickets. It is not fine for
60,000 support tickets — a linear scan over every single vector, on every single question, gets
slower in direct proportion to how much data you have. A **vector store** is a database purpose-built
to answer *"which stored vectors are closest to this one?"* without scanning everything, and to keep
that data around between runs instead of re-embedding it every time your script starts.

> 📌 **Worth knowing**
> - `langchain_chroma` is its own actively maintained package. **FAISS is not** — it still lives
>   inside `langchain_community`, the same sunset package flagged in `06_Document_Loaders`. Both work
>   fine for learning; know the difference if you're choosing one for something long-lived.
> - This notebook uses **Hugging Face's free local embeddings** throughout, so every cell runs without
>   an API key or a bill. Swap in `OpenAIEmbeddings` or `GoogleGenerativeAIEmbeddings` from
>   `01_Models` with a one-line change anywhere you see `embeddings` — the vector store code doesn't
>   change at all. That portability is the entire point of LangChain's `Embeddings` interface.

## Setup

In [7]:
# %pip install -q langchain langchain-core langchain-text-splitters python-dotenv
# %pip install -q langchain-huggingface sentence-transformers
# %pip install -q langchain-chroma                # Chroma — maintained, persistent
# %pip install -q "langchain-community==0.4.2" faiss-cpu   # FAISS — see the sunset note above
%pip install -q langchain-chroma faiss-cpu


Note: you may need to restart the kernel to use updated packages.


In [1]:
%pip list

Package                  Version
------------------------ -----------
accelerate               1.15.0
aiohappyeyeballs         2.7.1
aiohttp                  3.14.3
aiosignal                1.4.0
annotated-doc            0.0.5
annotated-types          0.8.0
anthropic                1.8.0
anyio                    4.15.1
asttokens                3.0.2
attrs                    26.1.0
beautifulsoup4           4.15.0
certifi                  2026.7.22
cffi                     2.1.1
charset-normalizer       3.5.1
click                    8.5.0
cloudpickle              3.1.2
colorama                 0.4.6
comm                     0.2.3
cryptography             50.0.1
dataclasses-json         0.6.7
debugpy                  1.8.22
diffusers                0.41.0.dev0
distro                   1.9.0
dnspython                2.8.0
docstring_parser         0.18.0
email-validator          2.3.0
executing                2.2.1
filelock                 4.0.1
filetype                 1.2.0
frozenlist   

In [6]:
%pip check

No broken requirements found.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os, shutil
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Environment ready.")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2286.98it/s]


Environment ready.


---
## Sample knowledge base

Same world as `06_Document_Loaders` and `07_Text_Splitters` — support policy documents, already
chunked and tagged with metadata. A vector store is where these chunks finally go to live.

In [9]:
chunks = [
    Document(page_content="Refunds are issued to the original payment method within 5-7 business days after we receive the item.",
             metadata={"department": "billing", "source": "returns_policy.txt"}),
    Document(page_content="Shipping fees are non-refundable unless the item arrived damaged or defective.",
             metadata={"department": "billing", "source": "returns_policy.txt"}),
    Document(page_content="Standard shipping takes 3-5 business days. Orders over $50 qualify for free standard shipping.",
             metadata={"department": "shipping", "source": "shipping_policy.txt"}),
    Document(page_content="International shipping takes 7-14 business days and is available to select countries only.",
             metadata={"department": "shipping", "source": "shipping_policy.txt"}),
    Document(page_content="To reset your password, choose 'Forgot password' on the sign-in page and follow the emailed link.",
             metadata={"department": "account", "source": "support_faq.csv"}),
    Document(page_content="Two-factor authentication codes are sent by SMS. Allow up to 2 minutes for delivery.",
             metadata={"department": "account", "source": "support_faq.csv"}),
]

print(f"{len(chunks)} chunks ready to store")


6 chunks ready to store


---
## 1. FAISS — fast, in-memory, file-based persistence

FAISS (Facebook AI Similarity Search) is a library, not a server — there's no database process to
run. It holds vectors in memory for very fast search, and you explicitly save/load it to disk when
you want it to survive between runs. Good default for prototyping, research, and anything that
doesn't need to be queried by more than one process at once.

In [10]:
from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(documents=chunks, embedding=embeddings)

print(type(faiss_store))
print("Vectors stored:", faiss_store.index.ntotal)


C:\Users\Deepak Kumar\AppData\Local\Temp\ipykernel_2592\1376953797.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


<class 'langchain_community.vectorstores.faiss.FAISS'>
Vectors stored: 6


In [11]:
results = faiss_store.similarity_search("How do I get my money back?", k=2)

for doc in results:
    print(doc.metadata, "->", doc.page_content)


{'department': 'billing', 'source': 'returns_policy.txt'} -> Refunds are issued to the original payment method within 5-7 business days after we receive the item.
{'department': 'account', 'source': 'support_faq.csv'} -> To reset your password, choose 'Forgot password' on the sign-in page and follow the emailed link.


Notice that query again: "money back," not "refund." This is the exact failure case from
`06_Document_Loaders` that broke plain keyword search — and it works here because the vector store is
searching by embedded *meaning*, not matching words.

In [12]:
# similarity_search_with_score — the same search, but with the actual distance returned.
# For FAISS's default index, LOWER means more similar (it's a distance, not a similarity percentage).
results_with_scores = faiss_store.similarity_search_with_score("How do I get my money back?", k=3)

for doc, score in results_with_scores:
    print(f"{score:.4f}  {doc.metadata['department']:<10}{doc.page_content[:60]}")


1.0744  billing   Refunds are issued to the original payment method within 5-7
1.4633  account   To reset your password, choose 'Forgot password' on the sign
1.6263  billing   Shipping fees are non-refundable unless the item arrived dam


In [13]:
# Persistence: save to disk, then load back in a FRESH variable — proving nothing was re-embedded.
FAISS_PATH = "faiss_support_kb"
faiss_store.save_local(FAISS_PATH)

reloaded_faiss = FAISS.load_local(
    FAISS_PATH, embeddings, allow_dangerous_deserialization=True,
)
print("Reloaded vectors:", reloaded_faiss.index.ntotal)
print(reloaded_faiss.similarity_search("password reset", k=1)[0].page_content)


Reloaded vectors: 6
To reset your password, choose 'Forgot password' on the sign-in page and follow the emailed link.


`allow_dangerous_deserialization=True` is required because loading uses Python's `pickle` under
the hood — only ever load a FAISS index you created yourself or trust completely, the same rule as
not running code you downloaded from a stranger.

---
## 2. Chroma — a persistent, filterable collection

Chroma behaves more like a lightweight database: a named **collection**, metadata you can filter on
directly in the query, and a `persist_directory` that saves automatically as you go — no explicit
"save" step required.

In [14]:
from langchain_chroma import Chroma

CHROMA_PATH = "chroma_support_kb"
shutil.rmtree(CHROMA_PATH, ignore_errors=True)   # clean slate for this notebook run

chroma_store = Chroma(
    collection_name="support_kb",
    embedding_function=embeddings,
    persist_directory=CHROMA_PATH,
)

chroma_store.add_documents(chunks)   # NOTE: plain add_documents — NOT the async aadd_documents()
                                       # unless you're inside an async function and actually await it.
print("Documents in collection:", chroma_store._collection.count())


Documents in collection: 6


> ⚠️ **A real mistake worth flagging:** `Chroma` also has `aadd_documents()` — the async version.
> Calling `chroma_store.aadd_documents(chunks)` *without* `await` doesn't raise an error; it silently
> returns an unfinished coroutine object and **adds nothing**. If a Chroma collection's count stays
> at 0 after you "add" documents, this is the first thing to check.

In [15]:
# The headline Chroma feature: filter by metadata DIRECTLY in the search, no post-filtering needed.
billing_only = chroma_store.similarity_search(
    "What happens if my payment doesn't go through?",
    k=3,
    filter={"department": "billing"},
)

for doc in billing_only:
    print(doc.metadata["department"], "->", doc.page_content)


billing -> Refunds are issued to the original payment method within 5-7 business days after we receive the item.
billing -> Shipping fees are non-refundable unless the item arrived damaged or defective.


In [16]:
# Without the filter, a shipping or account chunk could easily outrank a weaker billing match.
unfiltered = chroma_store.similarity_search(
    "What happens if my payment doesn't go through?", k=3,
)
print("Departments returned with NO filter:", [d.metadata["department"] for d in unfiltered])
print("Departments returned WITH filter   :", [d.metadata["department"] for d in billing_only])


Departments returned with NO filter: ['billing', 'billing', 'shipping']
Departments returned WITH filter   : ['billing', 'billing']


---
## 3. FAISS vs. Chroma — which one, when

| | FAISS | Chroma |
|---|---|---|
| Persistence | Manual (`save_local` / `load_local`) | Automatic, as you add documents |
| Metadata filtering | Supported, less ergonomic | First-class — a `filter` dict on every search |
| Best for | Fast prototyping, research, read-mostly workloads | Apps with ongoing writes, filtering, multiple sessions |
| Package status | Lives in unmaintained `langchain_community` | Actively maintained, own package |

**In practice:** reach for Chroma by default for anything you're actually going to keep building on
— the metadata filtering alone (Section 2) earns it for most real support/knowledge-base apps. FAISS
is a fine choice when you just need fast similarity search over a fixed, read-only set of vectors and
don't need per-query filtering.

---
## 4. Adding more documents later — incremental updates

Knowledge bases grow. Both stores support adding new documents to an existing store without rebuilding
it from scratch — this is the "I shipped v1, now a new policy exists" case.

In [17]:
new_policy = Document(
    page_content="Gift cards are refundable within 14 days of purchase if unused, directly to store credit.",
    metadata={"department": "billing", "source": "returns_policy.txt"},
)

chroma_store.add_documents([new_policy])
print("Documents in collection after update:", chroma_store._collection.count())

print(chroma_store.similarity_search("can I return a gift card", k=1, filter={"department": "billing"})[0].page_content)


Documents in collection after update: 7
Gift cards are refundable within 14 days of purchase if unused, directly to store credit.


---
## 5. Bridging to `09_Retriever` — `.as_retriever()`

Every vector store can convert itself into a `Retriever` — a `Runnable` that fits directly into the
chains from `03_Runnables`/`04_Chains` (`retriever | prompt | model | parser`). That conversion, and
everything a retriever can do beyond plain similarity search (MMR, multi-query, re-ranking), is the
whole next notebook — this is just the bridge.

In [18]:
retriever = chroma_store.as_retriever(search_kwargs={"k": 2})

docs = retriever.invoke("how long does international shipping take")
for d in docs:
    print(d.metadata, "->", d.page_content)


{'department': 'shipping', 'source': 'shipping_policy.txt'} -> International shipping takes 7-14 business days and is available to select countries only.
{'source': 'shipping_policy.txt', 'department': 'shipping'} -> Standard shipping takes 3-5 business days. Orders over $50 qualify for free standard shipping.


---
## Real-world capstone: a knowledge base that survives a restart — and proves it

The actual value of a vector store isn't the search itself (you already had that with a Python list
in `01_Models`) — it's that embedding thousands of documents is slow and costs money, and a vector
store means you pay that cost **once**, not on every run of your app. This capstone proves it:
build the store, "restart" by creating a brand new Python object pointed at the same directory, and
confirm the data — and the ability to search it — survived with zero re-embedding.

In [19]:
def build_knowledge_base(documents: list[Document], persist_directory: str, embeddings) -> Chroma:
    shutil.rmtree(persist_directory, ignore_errors=True)
    store = Chroma(collection_name="support_kb", embedding_function=embeddings, persist_directory=persist_directory)
    store.add_documents(documents)
    return store


KB_PATH = "capstone_kb"
store = build_knowledge_base(chunks, KB_PATH, embeddings)
print(f"Built: {store._collection.count()} documents, saved to '{KB_PATH}/'")

# --- simulate a restart: a brand new variable, a brand new Chroma client, same directory ---
del store

reopened_store = Chroma(collection_name="support_kb", embedding_function=embeddings, persist_directory=KB_PATH)
print(f"Reopened: {reopened_store._collection.count()} documents — no re-embedding happened")

results = reopened_store.similarity_search("my card got charged twice", k=2, filter={"department": "billing"})
for d in results:
    print(" ->", d.page_content)


Built: 6 documents, saved to 'capstone_kb/'
Reopened: 6 documents — no re-embedding happened
 -> Shipping fees are non-refundable unless the item arrived damaged or defective.
 -> Refunds are issued to the original payment method within 5-7 business days after we receive the item.


### Try it yourself
1. Add a `category` field to each chunk's metadata (e.g. `"policy"` vs. `"faq"`) and filter on it
   alongside `department`.
2. Time embedding and storing 500 synthetic documents vs. searching that same store 500 times —
   confirm search stays fast while re-embedding would not.
3. Try `faiss_store.delete(ids=[...])` — FAISS needs explicit IDs to remove a vector; find out what
   `chroma_store.delete(where={"department": "billing"})` does differently.
4. Swap `embeddings` for the OpenAI embeddings from `01_Models/02_embedding_models.ipynb` and rerun
   the capstone — confirm nothing except that one line changes.

---
**Next:** `09_Retriever/` — plain similarity search is the simplest way to query a vector store, not
the only one. MMR, multi-query expansion, and re-ranking all build on exactly the `chroma_store` and
`faiss_store` you just created.
